In [11]:
import os
import json
import torch
import numpy as np

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

In [ ]:
FINAL_DIR = "/Users/youhorng/Desktop/projects/multi-label-email-intent-classification/model/final_model"
META_PATH = "/Users/youhorng/Desktop/projects/multi-label-email-intent-classification/notebooks/preprocess_meta.json"

In [13]:
with open(META_PATH, "r") as f:
    meta = json.load(f)

BASE_MODEL_NAME = meta["model_name"]          # "distilbert-base-uncased"
MAX_LENGTH = meta.get("max_length", 256)

label2id = {k: int(v) for k, v in meta["label2id"].items()}
id2label = {int(k): v for k, v in meta["id2label"].items()}
num_labels = len(label2id)

print("Base model:", BASE_MODEL_NAME)
print("num_labels:", num_labels)
print("Labels:", id2label)

Base model: distilbert-base-uncased
num_labels: 10
Labels: {0: 'Business', 1: 'Customer Support', 2: 'Events & Invitations', 3: 'Finance & Bills', 4: 'Job Application', 5: 'Newsletters', 6: 'Personal', 7: 'Promotions', 8: 'Reminders', 9: 'Travel & Bookings'}


In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

Using device: mps


In [15]:
# 1. Tokenizer from your final_model folder (the one you saved)
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)

# 2. Base model with the SAME settings as training
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

# 3. Attach LoRA adapter (from local FINAL_DIR)
model = PeftModel.from_pretrained(base_model, FINAL_DIR)
model.to(device)
model.eval()

print("Model loaded with", model.config.num_labels, "labels.")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with 10 labels.


In [16]:
def predict_single(text: str, threshold: float = 0.55):
    """
    Run multilabel prediction on a single email text.
    Returns list of (label, probability) sorted by probability desc.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()

    results = [
        (id2label[i], float(p))
        for i, p in enumerate(probs)
        if p >= threshold
    ]

    # Sort high → low
    results = sorted(results, key=lambda x: x[1], reverse=True)

    # If nothing passes threshold, still return the top label
    if not results:
        top_idx = int(np.argmax(probs))
        results = [(id2label[top_idx], float(probs[top_idx]))]

    return results


In [17]:
from typing import List

def predict_batch(texts: List[str], threshold: float = 0.55):
    """
    Run multilabel prediction on a list of texts.
    Returns list of lists: [[(label, prob), ...], ...]
    """
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()

    batch_results = []
    for row in probs:
        results = [
            (id2label[i], float(p))
            for i, p in enumerate(row)
            if p >= threshold
        ]
        results = sorted(results, key=lambda x: x[1], reverse=True)
        if not results:
            top_idx = int(np.argmax(row))
            results = [(id2label[top_idx], float(row[top_idx]))]
        batch_results.append(results)

    return batch_results


In [23]:
email_text = """
Reminder: Upcoming Team Meeting Dear Team, Just a friendly reminder that our next team meeting is scheduled for this Friday at 10:00 AM in the conference room. Please make sure to review the agenda beforehand and come prepared to discuss your updates and action items. If you have any scheduling conflicts, please let me know as soon as possible. Looking forward to a productive meeting! Best regards, [Your Name]
"""

print(predict_single(email_text, threshold=0.55))

[('Reminders', 0.930374801158905), ('Business', 0.7522909045219421)]
